In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
df = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")

In [3]:
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [4]:
print(df.columns)

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')


In [5]:
df['Ticket'].isnull().sum()

np.int64(0)

In [6]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [7]:
df['Cabin']

0       NaN
1       C85
2       NaN
3      C123
4       NaN
       ... 
886     NaN
887     B42
888     NaN
889    C148
890     NaN
Name: Cabin, Length: 891, dtype: object

In [8]:
X = df.drop('Survived' , axis = 1)
y = df['Survived']

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
%%writefile feature_engineering.py
import pandas as pd

def add_features(df):
    df = df.copy()
    df.drop(['PassengerId', 'Ticket'], axis=1, inplace=True)
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df.drop(columns=['SibSp', 'Parch'], axis=1, inplace=True)
    df['FamilyGroup'] = pd.cut(df['FamilySize'], bins=[0, 1, 3, 11], labels=['Alone', 'Small', 'Large'])
    df.drop('FamilySize', axis=1, inplace=True)
    df['FamilyGroup'] = df['FamilyGroup'].map({'Alone': 0, 'Small': 1, 'Large': 2}).astype(int)
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
    df['Cabin'] = df['Cabin'].notna().astype(int)
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].apply(lambda x: x if x in ['Mr', 'Miss', 'Mrs', 'Master'] else 'Rare')
    df.drop('Name', axis=1, inplace=True)
    return df

Overwriting feature_engineering.py


In [11]:
!cat feature_engineering.py

import pandas as pd

def add_features(df):
    df = df.copy()
    df.drop(['PassengerId', 'Ticket'], axis=1, inplace=True)
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df.drop(columns=['SibSp', 'Parch'], axis=1, inplace=True)
    df['FamilyGroup'] = pd.cut(df['FamilySize'], bins=[0, 1, 3, 11], labels=['Alone', 'Small', 'Large'])
    df.drop('FamilySize', axis=1, inplace=True)
    df['FamilyGroup'] = df['FamilyGroup'].map({'Alone': 0, 'Small': 1, 'Large': 2}).astype(int)
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
    df['Cabin'] = df['Cabin'].notna().astype(int)
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].apply(lambda x: x if x in ['Mr', 'Miss', 'Mrs', 'Master'] else 'Rare')
    df.drop('Name', axis=1, inplace=True)
    return df


In [12]:
from feature_engineering import add_features
from sklearn.preprocessing import FunctionTransformer

feature_step = FunctionTransformer(add_features)

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier

preprocessor = ColumnTransformer([
    ('age_impute', SimpleImputer(strategy='mean'), ['Age']),
    ('embarked', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore'))
    ]), ['Embarked']),
    ('title_ohe', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Title']),
], remainder='passthrough')  # baaki columns (Sex already mapped, Pclass, etc.) as-is chale jayenge

full_pipeline = Pipeline([
    ('feature_engineering', feature_step),
    ('preprocessing', preprocessor),
    ('model', XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.2, random_state=42))
])

In [14]:
from sklearn.model_selection import GridSearchCV

params = {
    'model__n_estimators': [100, 200, 300],
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__max_depth': [3, 5, 7],
}

grid = GridSearchCV(full_pipeline, params, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)  # raw data, feature engineering pipeline ke andar hi ho jayegi

print("Best Params:", grid.best_params_)
print("Best Accuracy:", grid.best_score_)

Best Params: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 100}
Best Accuracy: 0.832837584950261


In [15]:
from sklearn.metrics import accuracy_score

y_pred = grid.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred))

Test Accuracy: 0.8044692737430168


In [16]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(grid.best_estimator_, X, y, cv=10)
print(scores.mean(), scores.std())

0.832784019975031 0.034179718645531695


In [17]:
final_model = grid.best_estimator_
final_model.fit(X, y)

import joblib
joblib.dump(final_model, 'titanic_pipeline.pkl')

['titanic_pipeline.pkl']

In [19]:
import os
print(os.listdir('/kaggle/working'))

['feature_engineering.py', '__pycache__', '.virtual_documents', 'titanic_pipeline.pkl']
